In [18]:
import re
import os
from collections import defaultdict
import numpy as np
import pandas as pd

In [19]:
## Use this for 1L sentences
folder_path = '/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences'
files = os.listdir(folder_path)
txt_files = [file for file in files if file.endswith('.txt')]
len(txt_files)

100

In [20]:
txt_files[99]

'kon_agriculture_set1.txt'

In [21]:
taggedData = []
for file in txt_files[:99]:
    with open("/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences/" + file, 'r', encoding='utf-8') as file:
        content = file.read()
    content = content.split("\n")
    content = content[1:]  # Skip the first line
    taggedDataFile = [line.split('\t')[1] for line in content if 'Value' not in line.split('\t')[1]]
    taggedData.append(taggedDataFile)
    
allTaggedData = [item for sublist in taggedData for item in sublist]
allTaggedData[:2]

['दुसर्\u200dया\\QT_QTO वेळांचेर\\N_NN लेगीत\\PSP आमी\\PR_PRP गेले\\V_VM_VF .\\RD_PUNC',
 'परततकच\\V_VM_VNF फोटे\\N_NN पळोवन\\V_VM_VNF दिसता\\V_VM_VF की\\CC_CC_UT कितें\\PR_PRQ ल्हारां\\N_NN आशिल्लीं\\V_VM_VF .\\RD_PUNC']

In [22]:
allTaggedData = list(set(allTaggedData))
len(allTaggedData)

98986

In [23]:
sentences = [sentence.strip().split() for sentence in allTaggedData]
word_tag_pairs = [[token.rsplit("\\", 1) for token in sentence] for sentence in sentences]

In [24]:
print(len(sentences))
print(len(word_tag_pairs))

98986
98986


In [25]:
# Create a dictionary of word-to-tag mappings
word_to_tag = defaultdict(lambda: "NN")  # Default to noun (NN)

In [26]:
filtered_sentences = []

for sentence in word_tag_pairs:
    filtered_sentence = [item for item in sentence if isinstance(item, (list, tuple)) and len(item) == 2]  # Keep only valid pairs
    
    if filtered_sentence:  # Add to filtered list if not empty
        filtered_sentences.append(filtered_sentence)


## Replace original list if needed
word_tag_pairs = filtered_sentences 

In [27]:
for sentence in word_tag_pairs:
    for word, tag in sentence:
        word_to_tag[word] = tag

In [28]:
# Define simple rule-based tagging function
def rule_based_pos_tagger(sentence):
    tokens = sentence.split()
    tagged_sentence = []
    
    for token in tokens:
        if re.match(r"^[0-9]+$", token):  # Rule for numbers
            tag = "CD"
        elif token in word_to_tag:
            tag = word_to_tag[token]
        elif token.endswith("ा") or token.endswith("ी") or token.endswith("े"):  # Rule for adjectives
            tag = "JJ"
        elif token.endswith("ने") or token.endswith("ली") or token.endswith("ले"):  # Rule for verbs
            tag = "V_VM_VF"
        else:
            tag = "NN"  # Default to noun
        
        tagged_sentence.append((token, tag))
    
    return tagged_sentence

In [29]:
# Example usage
print(rule_based_pos_tagger("ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवूच घेना ."))


[('ग्रेटर', 'NN'), ('नोयडा', 'N_NNP'), ('वेस्टाचे', 'JJ'), ('त्रास', 'N_NN'), ('कमी', 'JJ'), ('जावपाचें', 'V_VM_VNF'), ('नांवूच', 'N_NN'), ('घेना', 'V_VM_VNF'), ('.', 'RD_PUNC')]


In [30]:
testData = []
file_path = "/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences/kon_agriculture_set1.txt"

with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

content = content.split("\n")[1:]  # Skip the first line
testDataFile = [line.split('\t')[1] for line in content if line and 'Value' not in line.split('\t')[1]]
testData.append(testDataFile)

testTaggedData = [item for sublist in testData for item in sublist]
print(testTaggedData[:2])

['ग्रेटर\\RD_UNK नोयडा\\RD_UNK वेस्टाचे\\RD_UNK त्रास\\N_NN कमी\\QT_QTF जावपाचें\\V_VM_VNF नांवूच\\N_NN घेना\\V_VM_VF .\\RD_PUNC', 'आपल्यो\\PR_PRL मागण्यो\\N_NN घेवन\\V_VM_VNF निर्मीती\\JJ कार्य\\N_NN बंद\\RB करपाची\\V_VM_VNF शेतकार\\N_NN संघटणांत\\N_NN पैज\\N_NN लागल्या\\V_VM_VF .\\RD_PUNC']


In [31]:

print(len(testTaggedData))
ref_sentences = testTaggedData[:75]
print(len(ref_sentences))

1000
75


In [ ]:
#df_ref = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/ref_sentences.xlsx')
#df_ref.head()

In [ ]:
#ref_sentences = df_ref['ref_sentence'].tolist()
#print(len(ref_sentences))
#ref_sentences = ref_sentences[:75]
#print(len(ref_sentences))

In [32]:
df_test = pd.DataFrame(ref_sentences, columns=['actual_tags'])
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   actual_tags  75 non-null     object
dtypes: object(1)
memory usage: 728.0+ bytes


In [35]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)  # Split at last occurrence of '\'
            words.append(word)
            tags.append(tag)
    return words, tags

# Apply preprocessing to extract words & POS tags
df_test["sentence"], df_test["labels"] = zip(*df_test["actual_tags"].map(preprocess_sentence))
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   actual_tags  75 non-null     object
 1   sentence     75 non-null     object
 2   labels       75 non-null     object
dtypes: object(3)
memory usage: 1.9+ KB


In [36]:
df_test.head()

,actual_tags,sentence,labels
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,"[ग्रेटर, नोयडा, वेस्टाचे, त्रास, कमी, जावपाचें...","[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN..."
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,"[आपल्यो, मागण्यो, घेवन, निर्मीती, कार्य, बंद, ...","[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN..."
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,"[आयतरा, मायचा, गांवांत, शेतकार, संघर्श, समितीन...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_..."
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,"[लुकसाण, भरपाय, दिवपाच्या, नांवांचेर, शेतकारा,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,..."
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,"[पंचायतींत, शेतकारांनी, सांगलां, की, म्हामंडळ,...","[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N..."


In [37]:
def joinSen(words):
    sentence = " ".join(words)
    return sentence


df_test["joined_sent"] = df_test["sentence"].apply(joinSen)
df_test.head()

,actual_tags,sentence,labels,joined_sent
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,"[ग्रेटर, नोयडा, वेस्टाचे, त्रास, कमी, जावपाचें...","[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...",ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,"[आपल्यो, मागण्यो, घेवन, निर्मीती, कार्य, बंद, ...","[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...",आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,"[आयतरा, मायचा, गांवांत, शेतकार, संघर्श, समितीन...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...",आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,"[लुकसाण, भरपाय, दिवपाच्या, नांवांचेर, शेतकारा,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...",लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,"[पंचायतींत, शेतकारांनी, सांगलां, की, म्हामंडळ,...","[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...",पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...


In [38]:
df_test["output"] = df_test["joined_sent"].apply(rule_based_pos_tagger)
df_test.head()

,actual_tags,sentence,labels,joined_sent,output
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,"[ग्रेटर, नोयडा, वेस्टाचे, त्रास, कमी, जावपाचें...","[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...",ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...,"[(ग्रेटर, NN), (नोयडा, N_NNP), (वेस्टाचे, JJ),..."
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,"[आपल्यो, मागण्यो, घेवन, निर्मीती, कार्य, बंद, ...","[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...",आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...,"[(आपल्यो, PR_PRF), (मागण्यो, N_NN), (घेवन, V_V..."
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,"[आयतरा, मायचा, गांवांत, शेतकार, संघर्श, समितीन...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...",आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...,"[(आयतरा, JJ), (मायचा, JJ), (गांवांत, N_NN), (श..."
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,"[लुकसाण, भरपाय, दिवपाच्या, नांवांचेर, शेतकारा,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...",लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...,"[(लुकसाण, N_NN), (भरपाय, N_NN), (दिवपाच्या, V_..."
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,"[पंचायतींत, शेतकारांनी, सांगलां, की, म्हामंडळ,...","[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...",पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...,"[(पंचायतींत, N_NN), (शेतकारांनी, N_NN), (सांगल..."


In [41]:
df_test["joined_output"] = df_test["output"].apply(makeTagedSentences)
df_test.head()

,actual_tags,sentence,labels,joined_sent,output,joined_output
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,"[ग्रेटर, नोयडा, वेस्टाचे, त्रास, कमी, जावपाचें...","[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...",ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...,"[(ग्रेटर, NN), (नोयडा, N_NNP), (वेस्टाचे, JJ),...",ग्रेटर\NN नोयडा\N_NNP वेस्टाचे\JJ त्रास\N_NN क...
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,"[आपल्यो, मागण्यो, घेवन, निर्मीती, कार्य, बंद, ...","[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...",आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...,"[(आपल्यो, PR_PRF), (मागण्यो, N_NN), (घेवन, V_V...",आपल्यो\PR_PRF मागण्यो\N_NN घेवन\V_VM_VNF निर्म...
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,"[आयतरा, मायचा, गांवांत, शेतकार, संघर्श, समितीन...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...",आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...,"[(आयतरा, JJ), (मायचा, JJ), (गांवांत, N_NN), (श...",आयतरा\JJ मायचा\JJ गांवांत\N_NN शेतकार\N_NN संघ...
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,"[लुकसाण, भरपाय, दिवपाच्या, नांवांचेर, शेतकारा,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...",लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...,"[(लुकसाण, N_NN), (भरपाय, N_NN), (दिवपाच्या, V_...",लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,"[पंचायतींत, शेतकारांनी, सांगलां, की, म्हामंडळ,...","[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...",पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...,"[(पंचायतींत, N_NN), (शेतकारांनी, N_NN), (सांगल...",पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VF...


In [42]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['actual_tags'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_exp, mismatched_indices = add_length_columns(df_test)
print(mismatched_indices)

for ind in mismatched_indices:
    df_test = df_test.drop(ind)

[8]


In [45]:
# Apply function to each row
df_test[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_test.apply(lambda row: evaluate_tagging(row["actual_tags"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_test.head()

,actual_tags,sentence,labels,joined_sent,output,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,"[ग्रेटर, नोयडा, वेस्टाचे, त्रास, कमी, जावपाचें...","[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...",ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...,"[(ग्रेटर, NN), (नोयडा, N_NNP), (वेस्टाचे, JJ),...",ग्रेटर\NN नोयडा\N_NNP वेस्टाचे\JJ त्रास\N_NN क...,9,9,0.444444,0.388889,0.444444,0.407407,"[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...","[NN, N_NNP, JJ, N_NN, JJ, V_VM_VNF, N_NN, V_VM..."
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,"[आपल्यो, मागण्यो, घेवन, निर्मीती, कार्य, बंद, ...","[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...",आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...,"[(आपल्यो, PR_PRF), (मागण्यो, N_NN), (घेवन, V_V...",आपल्यो\PR_PRF मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,12,12,0.666667,0.666667,0.666667,0.666667,"[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...","[PR_PRF, N_NN, V_VM_VNF, N_NN, N_NN, RB, V_VM_..."
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,"[आयतरा, मायचा, गांवांत, शेतकार, संघर्श, समितीन...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...",आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...,"[(आयतरा, JJ), (मायचा, JJ), (गांवांत, N_NN), (श...",आयतरा\JJ मायचा\JJ गांवांत\N_NN शेतकार\N_NN संघ...,17,17,0.647059,0.698529,0.647059,0.671280,"[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...","[JJ, JJ, N_NN, N_NN, N_NN, N_NN, N_NN, V_VM_VN..."
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,"[लुकसाण, भरपाय, दिवपाच्या, नांवांचेर, शेतकारा,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...",लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...,"[(लुकसाण, N_NN), (भरपाय, N_NN), (दिवपाच्या, V_...",लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,21,21,0.809524,0.904762,0.809524,0.852381,"[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, PSP, N_NNP,..."
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,"[पंचायतींत, शेतकारांनी, सांगलां, की, म्हामंडळ,...","[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...",पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...,"[(पंचायतींत, N_NN), (शेतकारांनी, N_NN), (सांगल...",पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VF...,10,10,0.600000,0.800000,0.600000,0.675000,"[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...","[N_NN, N_NN, V_VM_VF, CC_CCS, N_NNP, JJ, N_NN,..."


In [46]:
df_test.to_excel("test_rule_based_75_output.xlsx", index=False)

In [47]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_test["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_test["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      0.96      0.98        28
      CC_CCS       0.89      0.85      0.87        20
          CD       0.00      0.00      0.00         0
      DM_DMD       0.96      0.70      0.81        37
      DM_DMI       0.00      0.00      0.00         1
      DM_DMQ       0.00      0.00      0.00         0
      DM_DMR       0.50      0.50      0.50         2
          JJ       0.48      0.61      0.54        87
          NN       0.00      0.00      0.00         0
        N_NN       0.87      0.79      0.83       407
       N_NNP       0.24      0.40      0.30        10
       N_NST       0.91      0.56      0.69        18
      PR_PRC       0.00      0.00      0.00         0
      PR_PRF       0.80      0.80      0.80         5
      PR_PRI       1.00      1.00      1.00         1
      PR_PRL       0.83      0.83      0.83         6
      PR_PRP       0.23      1.00      0.38         3
  

In [48]:
df_ref = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/ref_sentences_Mannual.xlsx')
df_ref.head()

,sentences,ref_sentences
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...


In [49]:
df_ref["output"] = df_ref["sentences"].apply(rule_based_pos_tagger)
df_ref.head()

,sentences,ref_sentences,output
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,"[(मनशाच्या, N_NN), (शरिरांत, N_NN), (206, CD),..."
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,"[(मनशाचें, N_NN), (काळीज, N_NN), (दिसाक, N_NN)..."
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,"[(सूर्य, N_NN), (पृथ्वी, N_NN), (परस, PSP), (3..."
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,"[(नील, N_NNP), (आर्मस्ट्राँग, NN), (हो, DM_DMD..."
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,"[(पृथ्वीचो, N_NN), (सगल्यांत, QT_QTF), (व्हडलो..."


In [50]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

'सहस्त्रधारा\\N_NNP ही\\DM_DMD देहरादूनाची\\N_NNP पळोवपाची\\V_VM_VNF मुखेल\\JJ सुवात\\N_NN .\\RD_PUNC'

In [51]:
df_ref["joined_output"] = df_ref["output"].apply(makeTagedSentences)
df_ref.head()

,sentences,ref_sentences,output,joined_output
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,"[(मनशाच्या, N_NN), (शरिरांत, N_NN), (206, CD),...",मनशाच्या\N_NN शरिरांत\N_NN 206\CD हाडां\N_NN आ...
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,"[(मनशाचें, N_NN), (काळीज, N_NN), (दिसाक, N_NN)...",मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\QT_QT...
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,"[(सूर्य, N_NN), (पृथ्वी, N_NN), (परस, PSP), (3...",सूर्य\N_NN पृथ्वी\N_NN परस\PSP 3\CD लाख\QT_QTF...
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,"[(नील, N_NNP), (आर्मस्ट्राँग, NN), (हो, DM_DMD...",नील\N_NNP आर्मस्ट्राँग\NN हो\DM_DMD 1969\CD वर...
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,"[(पृथ्वीचो, N_NN), (सगल्यांत, QT_QTF), (व्हडलो...",पृथ्वीचो\N_NN सगल्यांत\QT_QTF व्हडलो\JJ आनी\CC...


In [52]:
df_ref.drop(["output"], axis=1, inplace=True)
df_ref

,sentences,ref_sentences,joined_output
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,मनशाच्या\N_NN शरिरांत\N_NN 206\CD हाडां\N_NN आ...
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\QT_QT...
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,सूर्य\N_NN पृथ्वी\N_NN परस\PSP 3\CD लाख\QT_QTF...
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,नील\N_NNP आर्मस्ट्राँग\NN हो\DM_DMD 1969\CD वर...
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,पृथ्वीचो\N_NN सगल्यांत\QT_QTF व्हडलो\JJ आनी\CC...
...,...,...,...
96,सकाळीं रेंदेर माडाचेर चडटात आनी माडाक बांदिल्ल...,सकाळीं\N_NN रेंदेर\N_NN माडाचेर\N_NN चडटात\V_V...,सकाळीं\N_NN रेंदेर\N_NN माडाचेर\NN चडटात\V_VM_...
97,तातूंत सूर एकठांय जाल्लो आसता .,तातूंत\PR_PRL सूर\N_NN एकठांय\RB जाल्लो\V_VM_V...,तातूंत\DM_DMD सूर\N_NN एकठांय\RB जाल्लो\V_VM_V...
98,माडाची सूर पितात लेगीत .,माडाची\N_NN सूर\N_NN पितात\V_VM_VF लेगीत\PSP ....,माडाची\JJ सूर\N_NN पितात\V_VAUX_VF लेगीत\PSP ....
99,"ह्या सुराचें विनाग्र करतात तशेंच सोरोय करतात, ...",ह्या\DM_DMD सुराचें\N_NN विनाग्र\N_NN करतात\V_...,ह्या\DM_DMD सुराचें\NN विनाग्र\N_NN करतात\V_VM...


In [53]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [54]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['ref_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_exp, mismatched_indices = add_length_columns(df_ref)
mismatched_indices

[43, 55, 56, 75, 80, 92, 99, 100]

In [55]:
for ind in mismatched_indices:
    df_ref = df_ref.drop(ind)

In [56]:
df_ref.info()

<class 'pandas.core.frame.DataFrame'>
Index: 93 entries, 0 to 98
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sentences             93 non-null     object
 1   ref_sentences         93 non-null     object
 2   joined_output         93 non-null     object
 3   ref_sentences_length  93 non-null     int64 
 4   joined_sent_length    93 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 4.4+ KB


In [57]:
# Apply function to each row
df_ref[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_ref.apply(lambda row: evaluate_tagging(row["ref_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_ref.head()

,sentences,ref_sentences,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,मनशाच्या\N_NN शरिरांत\N_NN 206\CD हाडां\N_NN आ...,6,6,0.833333,0.833333,0.833333,0.833333,"[N_NN, N_NN, QT_QTC, N_NN, V_VM_VF, RD_PUNC]","[N_NN, N_NN, CD, N_NN, V_VM_VF, RD_PUNC]"
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\QT_QT...,10,10,0.700000,0.800000,0.700000,0.745455,"[N_NN, N_NN, N_NN, RB, QT_QTC, N_NN, N_NN, N_N...","[N_NN, N_NN, N_NN, QT_QTF, CD, QT_QTF, N_NN, N..."
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,सूर्य\N_NN पृथ्वी\N_NN परस\PSP 3\CD लाख\QT_QTF...,20,20,0.600000,0.459524,0.600000,0.513333,"[N_NNP, N_NNP, PSP, QT_QTC, N_NN, PSP, QT_QTF,...","[N_NN, N_NN, PSP, CD, QT_QTF, PSP, QT_QTF, N_N..."
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,नील\N_NNP आर्मस्ट्राँग\NN हो\DM_DMD 1969\CD वर...,12,12,0.666667,0.916667,0.666667,0.744048,"[N_NNP, N_NNP, PR_PRP, N_NN, N_NN, N_NNP, N_NN...","[N_NNP, NN, DM_DMD, CD, N_NN, NN, N_NN, V_VM_V..."
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,पृथ्वीचो\N_NN सगल्यांत\QT_QTF व्हडलो\JJ आनी\CC...,10,10,0.700000,0.800000,0.700000,0.733333,"[N_NNP, RP_INTF, JJ, CC_CCD, JJ, JJ, N_NN, CC_...","[N_NN, QT_QTF, JJ, CC_CCD, JJ, JJ, NN, CC_CCS,..."


In [58]:
df_ref.to_excel("test_rule_based_100_manual_output.xlsx", index=False)

In [59]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_ref["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_ref["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        20
      CC_CCS       0.86      0.95      0.90        20
          CD       0.00      0.00      0.00         0
      DM_DMD       0.79      0.90      0.84        30
      DM_DMI       0.33      1.00      0.50         1
      DM_DMR       0.00      0.00      0.00         2
          JJ       0.77      0.90      0.83        60
          NN       0.00      0.00      0.00         0
        N_NN       0.95      0.79      0.87       345
       N_NNP       0.80      0.79      0.80        62
       N_NST       0.69      0.61      0.65        18
      PR_PRF       1.00      1.00      1.00         6
      PR_PRL       0.00      0.00      0.00         2
      PR_PRP       0.50      0.50      0.50         8
         PSP       0.87      0.84      0.86        32
      QT_QTC       1.00      0.50      0.67        28
      QT_QTF       0.49      0.85      0.62        20
  

In [65]:
df_anwani = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/anwani_ref_sentences.xlsx')
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,Unnamed: 3
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,NaN
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,NaN
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,NaN
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,NaN
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,NaN


In [66]:
df_anwani.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   sentences_cleaned    100 non-null    object
 1   predicted_sentences  100 non-null    object
 2   tagged_sentences     100 non-null    object
 3   Unnamed: 3           1 non-null      object
dtypes: object(4)
memory usage: 3.2+ KB


In [67]:
df_anwani["output"] = df_anwani["sentences_cleaned"].apply(rule_based_pos_tagger)
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,Unnamed: 3,output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,NaN,"[(आयकलां, V_VM_VF), (?, RD_PUNC), (लागीं, PSP)..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,NaN,"[(दोन, QT_QTC), (तीन, QT_QTC), (दिसांनी, N_NN)..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,NaN,"[(आनी, CC_CCD), (जाली, V_VM_VF), (ना, RP_NEG),..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,NaN,"[(हांव, PR_PRP), (अशें, DM_DMD), (आसा, V_VM_VF..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,NaN,"[(पूण, CC_CCS), (हांगा, N_NST), (रावनय, NN), (..."


In [68]:
df_anwani["joined_output"] = df_anwani["output"].apply(makeTagedSentences)
df_anwani.drop(["output"], axis=1, inplace=True)
df_anwani.drop(["Unnamed: 3"], axis=1, inplace=True)
df_anwani.drop(["predicted_sentences"], axis=1, inplace=True)
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\PSP ना\RP_NEG आ...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\NN जावचें\V_VM_VF...


In [72]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_anwani, mismatched_indices = add_length_columns(df_anwani)
mismatched_indices

[48, 49]

In [73]:
for ind in mismatched_indices:
    df_anwani = df_anwani.drop(ind)

df_anwani.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98 entries, 0 to 99
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sentences_cleaned     98 non-null     object
 1   tagged_sentences      98 non-null     object
 2   joined_output         98 non-null     object
 3   ref_sentences_length  98 non-null     int64 
 4   joined_sent_length    98 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 4.6+ KB


In [74]:
# Apply function to each row
df_anwani[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_anwani.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\PSP ना\RP_NEG आ...,14,14,0.642857,0.607143,0.642857,0.619048,"[V_VM_VF, RD_PUNC, N_NST, RP_NEG, DM_DMD, N_NN...","[V_VM_VF, RD_PUNC, PSP, RP_NEG, PR_PRP, N_NN, ..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,11,11,0.818182,0.909091,0.818182,0.854545,"[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, RD_...","[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, RD_..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,22,22,0.909091,0.909091,0.909091,0.909091,"[CC_CCD, V_VM_VNF, RP_NEG, CC_CCS, RD_PUNC, PR...","[CC_CCD, V_VM_VF, RP_NEG, CC_CCS, RD_PUNC, PR_..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,17,17,0.764706,0.764706,0.764706,0.764706,"[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM...","[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\NN जावचें\V_VM_VF...,15,15,0.800000,0.900000,0.800000,0.831111,"[CC_CCS, N_NST, V_VM_VNF, V_VM_VNF, RP_NEG, RD...","[CC_CCS, N_NST, NN, V_VM_VF, RP_NEG, RD_PUNC, ..."


In [75]:
df_anwani.to_excel("test_rule_based_100_anwani_output.xlsx", index=False)

In [76]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_anwani["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_anwani["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        13
      CC_CCS       0.56      0.93      0.70        43
      DM_DMD       0.80      0.83      0.81        52
      DM_DMI       0.70      0.70      0.70        10
      DM_DMQ       1.00      0.07      0.12        15
          JJ       0.22      0.83      0.34        12
          NN       0.00      0.00      0.00         0
         NST       0.00      0.00      0.00         1
        N_NN       0.86      0.83      0.84       201
       N_NNP       0.00      0.00      0.00         1
       N_NST       0.81      0.73      0.77        78
      PR_PRF       1.00      1.00      1.00         1
      PR_PRI       1.00      0.36      0.53        11
      PR_PRL       0.00      0.00      0.00         0
      PR_PRP       0.92      0.92      0.92       131
      PR_PRQ       0.56      0.45      0.50        20
         PSP       0.53      0.55      0.54        38
  